# Search V2 Test Runner

Scratch notebook for exercising the V2 search pipeline one stage at a time on a single query. Currently wired up to run Step 0 (flow routing) in isolation; more stages can be added as they come online.

## Cell 1 — Setup, imports, DB readiness

Run once. Opens the Postgres pool, initializes Redis, and pings Qdrant (idempotent — safe to rerun). Every import used anywhere below lives here.

In [1]:
import sys, time
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Pipeline steps
from search_v2.step_0 import run_step_0
from search_v2.step_1 import run_step_1
from search_v2.steps_0_2_orchestrator import run_steps_0_to_2

# Shared schemas
from schemas.step_0_flow_routing import Step0Response
from schemas.step_1 import Step1Response
from schemas.step_2 import Step2Response
from schemas.enums import SearchFlow

# DB
from db.postgres import pool as postgres_pool
from db.qdrant import check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

# DB readiness (idempotent — mirrors test_stage_1_to_4.ipynb)
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")


/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


## Cell 2 — Configuration

Set the query and LLM preset. Uncomment one preset block or assign the three variables manually.

In [2]:
# ---- Query ----
query = "movies my dad would love"

# ---- Presets (uncomment one) ----
# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
# provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Shared ----
today = date.today()

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")


Query:    movies my dad would love
Provider: gemini
Model:    gemini-3-flash-preview
Kwargs:   {'thinking_config': {'thinking_budget': 0}}
Today:    2026-04-24


## Cell 3 — Step 1: Spin Generation (isolated)

Runs the raw query through Step 1's spin generator in isolation. Prints the decomposition (hard commitments / soft interpretations / open dimensions), the original-query label, and the two spins with their levers and queries.

In [16]:
query = "dogshit comedies to watch while high"

start = time.perf_counter()
step_1_response, step_1_in_tok, step_1_out_tok, step_1_call_elapsed = await run_step_1(query)
step_1_elapsed = time.perf_counter() - start

print(f"Step 1 completed in {step_1_elapsed:.2f}s (LLM call: {step_1_call_elapsed:.2f}s)")
print(f"Tokens — input: {step_1_in_tok:,}  output: {step_1_out_tok:,}\n")

print("DECOMPOSITION")
print("-" * 72)
print(f"  hard_commitments     ({len(step_1_response.hard_commitments)}): {step_1_response.hard_commitments}")
print(f"  soft_interpretations ({len(step_1_response.soft_interpretations)}): {step_1_response.soft_interpretations}")
print(f"  open_dimensions      ({len(step_1_response.open_dimensions)}): {step_1_response.open_dimensions}")
print(f"  original_query_label: {step_1_response.original_query_label!r}")
print()

print("SPINS")
print("-" * 72)
for i, spin in enumerate(step_1_response.spins, start=1):
    print(f"  Spin {i} — {spin.ui_label!r}")
    print(f"    branching_opportunity: {spin.branching_opportunity}")
    print(f"    distinctness:          {spin.distinctness}")
    print(f"    query:                 {spin.query!r}")
    print()

print("FULL JSON")
print("-" * 72)
print(step_1_response.model_dump_json(indent=2))


Step 1 completed in 2.76s (LLM call: 2.76s)
Tokens — input: 3,360  output: 373

DECOMPOSITION
------------------------------------------------------------------------
  hard_commitments     (1): ['comedies']
  soft_interpretations (2): ['dogshit', 'watch while high']
  open_dimensions      (2): ['era', 'sub-genre like stoner or parody']
  original_query_label: 'Low-Brow Comedies'

SPINS
------------------------------------------------------------------------
  Spin 1 — "So-Bad-It's-Good Disasters"
    branching_opportunity: Reinterprets 'dogshit' (soft interpretation) as 'so-bad-it's-good' cult failures rather than just generic low-quality movies.
    distinctness:          Focuses on notorious cinematic disasters and bizarre cult flops like 'The Room' or 'Mac and Me' that offer unintentional humor, whereas the original might just return poorly reviewed mainstream comedies.
    query:                 "so-bad-it's-good cult failure comedies to watch while high"

  Spin 2 — 'Trippy Surre

## Cell 4 — Step 2: Query Pre-pass (isolated)

Runs the raw query through Step 2's pre-pass LLM in isolation. Prints the overall intention exploration, every attribute fragment with its nested modifiers (polarity / role markers) and coverage evidence (captured meaning → category → fit → atomic rewrite).

In [3]:
from search_v2.step_2 import run_step_2

query = "horror movies that aren't too scary or depressing"

start = time.perf_counter()
step_2_response, step_2_in_tok, step_2_out_tok, step_2_call_elapsed = await run_step_2(query)
step_2_elapsed = time.perf_counter() - start

print(f"Step 2 completed in {step_2_elapsed:.2f}s (LLM call: {step_2_call_elapsed:.2f}s)")
print(f"Tokens — input: {step_2_in_tok:,}  output: {step_2_out_tok:,}\n")

print("OVERALL QUERY INTENTION")
print("-" * 72)
print(f"  {step_2_response.overall_query_intention_exploration}")
print()

print(f"REQUIREMENTS ({len(step_2_response.requirements)})")
print("-" * 72)
for i, frag in enumerate(step_2_response.requirements, start=1):
    print(f"  [{i}] {frag.query_text!r}")
    print(f"      description: {frag.description}")
    if frag.modifiers:
        print(f"      modifiers ({len(frag.modifiers)}):")
        for m in frag.modifiers:
            print(f"        - [{m.type.value}] {m.original_text!r} — {m.effect}")
    if frag.coverage_evidence:
        print(f"      coverage_evidence ({len(frag.coverage_evidence)}):")
        for j, ev in enumerate(frag.coverage_evidence, start=1):
            print(f"        {j}. captured_meaning: {ev.captured_meaning}")
            print(f"           category:         {ev.category_name.value}")
            print(f"           fit_quality:      {ev.fit_quality}")
            print(f"           atomic_rewrite:   {ev.atomic_rewrite!r}")
    print()

print("FULL JSON")
print("-" * 72)
print(step_2_response.model_dump_json(indent=2))

Step 2 completed in 3.81s (LLM call: 3.80s)
Tokens — input: 7,755  output: 448

OVERALL QUERY INTENTION
------------------------------------------------------------------------
  The user is seeking horror films with specific tonal and intensity constraints, likely for a more accessible or 'light' viewing experience within the genre. They are explicitly filtering for moderate levels of fear and emotional weight, avoiding the extreme ends of horror's psychological or visceral spectrum.

REQUIREMENTS (3)
------------------------------------------------------------------------
  [1] 'horror'
      description: Specifies the primary genre for the search.
      coverage_evidence (1):
        1. captured_meaning: The user is looking for films within the broad horror genre.
           category:         Top-level genre
           fit_quality:      clean
           atomic_rewrite:   'horror genre'

  [2] 'scary'
      description: Specifies the intensity of the fear response or tension.
      m

## Cell 4 — Steps 0-2 Orchestrator

Runs steps 0 and 1 in parallel against the raw query, then dispatches per the routing decision. The standard flow fans out into 1-3 step-2 branches (3 minus the number of non-standard flows that fired). Exact-title and similarity flows are TODO placeholders today; the orchestrator surfaces whether each WOULD have run.

In [ ]:
orch_result = await run_steps_0_to_2(query)

print(f"Total elapsed: {orch_result.total_elapsed:.2f}s\n")

print("STEP 0")
print("-" * 72)
print(f"  elapsed: {orch_result.step0_elapsed:.2f}s  tokens in/out: {orch_result.step0_input_tokens:,} / {orch_result.step0_output_tokens:,}")
etf = orch_result.step0_response.exact_title_flow_data
sim = orch_result.step0_response.similarity_flow_data
print(f"  exact_title  should_be_searched={etf.should_be_searched}  title={etf.exact_title_to_search!r}")
print(f"  similarity   should_be_searched={sim.should_be_searched}  title={sim.similar_search_title!r}")
print(f"  enable_primary_flow={orch_result.step0_response.enable_primary_flow}  primary_flow={orch_result.step0_response.primary_flow.value}")
print()

print("STEP 1")
print("-" * 72)
if orch_result.step1_response is None:
    print(f"  FAILED: {orch_result.step1_error}")
else:
    s1 = orch_result.step1_response
    print(f"  elapsed: {orch_result.step1_elapsed:.2f}s  tokens in/out: {orch_result.step1_input_tokens:,} / {orch_result.step1_output_tokens:,}")
    print(f"  original_query_label: {s1.original_query_label!r}")
    for i, spin in enumerate(s1.spins, start=1):
        print(f"  spin {i}: {spin.ui_label!r}  query={spin.query!r}")
print()

print("FLOW DISPATCH")
print("-" * 72)
print(f"  exact_title flow would execute: {orch_result.exact_title_flow_executed}  (TODO — not yet implemented)")
print(f"  similarity  flow would execute: {orch_result.similarity_flow_executed}  (TODO — not yet implemented)")
print(f"  step 2 branches planned: {len(orch_result.step2_branches)}")
print()

if orch_result.step2_branches:
    print("STEP 2 BRANCHES")
    print("-" * 72)
    for branch in orch_result.step2_branches:
        print(f"  [{branch.kind}] {branch.ui_label!r}  query={branch.query!r}")
        if branch.error is not None:
            print(f"    FAILED: {branch.error}")
        else:
            print(f"    elapsed: {branch.elapsed:.2f}s  tokens in/out: {branch.input_tokens:,} / {branch.output_tokens:,}")
            print(f"    overall_intention: {branch.response.overall_query_intention_exploration}")
            print(f"    requirements ({len(branch.response.requirements)}):")
            for req in branch.response.requirements:
                mods = ""
                if req.modifiers:
                    mods = "  modifiers=[" + ", ".join(
                        f"{m.type.value}:{m.original_text!r}" for m in req.modifiers
                    ) + "]"
                cats = ", ".join(ev.category_name.value for ev in req.coverage_evidence)
                print(f"      - {req.query_text!r}{mods}  → [{cats}]")
        print()
